In [1]:
from pathlib import Path
import pandas as pd

root = Path("virus-amplicon/output").resolve()
pos = 28270
out = Path("snvs_28270.csv").resolve()

def extract_strain_sample(root: Path, snvs_path: Path):
    # Expected structure: root/strain/sample/snvs.csv
    rel = snvs_path.relative_to(root)
    parts = rel.parts
    if len(parts) < 3:
        return ("UNKNOWN_STRAIN", "UNKNOWN_SAMPLE")
    return parts[0], parts[1]

def row_has_28270(df: pd.DataFrame, pos: int) -> pd.Series:
    s = df.astype(str)
    return s.apply(
        lambda row: row.str.contains(rf"\b{pos}\b", regex=True, na=False).any(),
        axis=1,
    )

snvs_files = sorted(root.rglob("snvs.csv"))
print(f"Found {len(snvs_files)} snvs.csv files")

all_hits = []

for f in snvs_files:
    try:
        df = pd.read_csv(f)
    except Exception as e:
        print(f"[WARN] Failed to read {f}: {e}")
        continue

    if df.empty:
        continue

    strain, sample = extract_strain_sample(root, f)

    # Try to find an obvious position column first
    candidate_cols = [
        c for c in df.columns
        if c.lower() in {"pos", "position", "ref_pos", "ref_position", "site"}
    ]

    if candidate_cols:
        col = candidate_cols[0]
        try:
            mask = pd.to_numeric(df[col], errors="coerce") == pos
        except Exception:
            mask = df[col].astype(str).str.contains(rf"\b{pos}\b", regex=True, na=False)
    else:
        mask = row_has_28270(df, pos)

    hits = df.loc[mask].copy()
    if hits.empty:
        continue

    hits.insert(0, "strain", strain)
    hits.insert(1, "sample", sample)
    hits.insert(2, "source_path", str(f))

    all_hits.append(hits)

if not all_hits:
    print(f"No rows containing {pos} found.")
else:
    merged = pd.concat(all_hits, ignore_index=True)
    merged.to_csv(out, index=False)
    print(f"Wrote {len(merged)} rows to {out}")

merged.head() if all_hits else None

Found 13 snvs.csv files
Wrote 13 rows to /restricted/projectnb/challenge2025/sgRNAtor/jsprince/snvs_28270.csv


,strain,sample,source_path,Accession,NT.ID,NT.position,Frequency,InDel,NT.ref,NT.alt
0,alpha,AA0000144,/restricted/projectnb/challenge2025/sgRNAtor/j...,MN908947.3,28270-TA-T,28270,0.976794,True,TA,T
1,deltaA,AY.103,/restricted/projectnb/challenge2025/sgRNAtor/j...,MN908947.3,28270-TA-T,28270,0.987040,True,TA,T
2,deltaB,B.1.617.2,/restricted/projectnb/challenge2025/sgRNAtor/j...,MN908947.3,28270-TA-T,28270,0.987618,True,TA,T
3,gamma,gamma,/restricted/projectnb/challenge2025/sgRNAtor/j...,MN908947.3,28270-T-A,28270,0.024339,False,T,A
4,persistent,SRR31567433,/restricted/projectnb/challenge2025/sgRNAtor/j...,MN908947.3,28270-TAA-T,28270,0.917362,True,TAA,T
